# COS30049 A2 - The test tier of the accepted DS1 training stages, and the selection rule (Kaggle CPU session)

Serves `docs/TRAINING_PLAN.md` section 5 ("every accepted candidate once"): with `config/settings.json -> stageModels`
naming each method's accepted stage (C3 LOF, C4 K-Means, C5 DBSCAN, C6 the regression range variant), the DS1
features (with F9, an input of the K-Means winner) and the plants are rebuilt, then every method is scored and
evaluated on the REPORTING plants (seeds 30049-30053) with the existing steps: `scores`, `evaluate` (stability,
sanity and the A1 selection rule: stage C10), `significance` (per-plant hits) and `ds2-check`. `stage-test` saves
this tier's snapshot and gates it against M0's (committed in `reports/tables/stage_test/M0_*`), paired by plant with
an address-level bootstrap. CPU only, so no GPU quota.

**Settings:** Accelerator **None**, **Internet on**. Then **Save Version -> Save & Run All (Commit)**.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "ccc5c35b578d49100ef7e37cfbd90fddba1228a1"   # the code this run uses
SNAPSHOT = "C"        # this tier's snapshot name (stage_test/C_*.csv)
METHODS = []          # the accepted stages under test (29 Oct: lof, kmeans, dbscan, regression_interval); [] = only
                      # rebuild every table with the methods as they stand (no snapshot, no gate)
STEPS = ["features", "features --as-written", "split", "inject", "scores", "evaluate", "significance", "ds2-check"]
PINS = ["numpy==2.2.6", "pandas==2.2.3", "scikit-learn==1.6.1", "scipy==1.17.1", "pyarrow==20.0.0", "joblib==1.4.2"]

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

In [ ]:
def cli(*args, log):
    sh(sys.executable, "-m", "explorer.cli", *args, log=log)


for step in STEPS:
    t = time.time()
    cli(*step.split(), log=f"{step.split()[0]}{'-as-written' if '--as-written' in step else ''}.log")
    print(f"{step}: {(time.time() - t) / 60:.1f} min")
if METHODS:
    cli("stage-test", "--snapshot", SNAPSHOT, log="stage-test.log")
    cli("stage-test", "--before", "M0", "--after", SNAPSHOT, "--stage", f"{SNAPSHOT}-test", "--methods", *METHODS,
        log="stage-test.log")

In [ ]:
# Everything the steps rewrote (to commit), and the logs.
for rel in ["reports/tables", "models/selection.json", "data/processed/esplora/features.csv",
            "data/processed/esplora/features_as_written.csv", "data/processed/esplora/scores.csv", "reports/logs"]:
    src = SRC / rel
    dest = WORK / "stage" / rel
    if src.is_dir():
        shutil.copytree(src, dest, dirs_exist_ok=True)
    elif src.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dest)
(WORK / "kaggle_run.json").write_text(json.dumps({"commit": COMMIT, "steps": STEPS, "methods": METHODS,
    "hours": round((time.time() - T0) / 3600, 2),
    "packages": {p.split("==")[0]: version(p.split("==")[0]) for p in PINS}}, indent=2), encoding="utf-8")
print((SRC / "reports" / "tables" / "stages.csv").read_text(encoding="utf-8"))